# A01 — Full Results & Dataset Statistics
Covers: `tab:appendix-datasets`, `tab:appendix-full-seen`  (Appendix B)


In [1]:
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import appendix_viz_helpers as viz
importlib.reload(viz)

apply_style    = viz.apply_style
load_csv       = viz.load_csv
annotate_demo  = viz.annotate_demo
clean_axes     = viz.clean_axes
dataset_label  = viz.dataset_label
panel_label    = viz.panel_label
PALETTE        = viz.PALETTE
DATASET_LABELS = viz.DATASET_LABELS
apply_style()


In [2]:
stats   = load_csv('appendix_dataset_stats.csv')
results = load_csv('appendix_full_results_long.csv')
stats['dataset_label']   = stats['dataset'].map(dataset_label)
results['dataset_label'] = results['dataset'].map(dataset_label)
print(f"stats: {stats.shape}  results: {results.shape}")
print("stats cols:   ", list(stats.columns))
print("results cols: ", list(results.columns)[:8])
print("models:       ", sorted(results['model'].unique()))
print("metrics:      ", sorted(results['metric'].unique()))


stats: (4, 8)  results: (360, 8)
stats cols:    ['dataset', 'interactions', 'sessions', 'items', 'avg_session_len', 'data_status', 'data_note', 'dataset_label']
results cols:  ['dataset', 'model', 'split', 'metric', 'value', 'data_status', 'data_note', 'dataset_label']
models:        ['BERT4Rec', 'BPR', 'GRU4Rec', 'RouteRec', 'SASRec']
metrics:       ['hit@10', 'hit@20', 'hit@5', 'mrr@10', 'mrr@20', 'mrr@5', 'ndcg@10', 'ndcg@20', 'ndcg@5']


In [3]:
# ── Table B1 — Dataset statistics ────────────────────────────────────────────
disp_stats = (
    stats[['dataset_label', 'interactions', 'sessions', 'items', 'avg_session_len']]
    .rename(columns={
        'dataset_label':   'Dataset',
        'interactions':    'Interactions',
        'sessions':        'Sessions',
        'items':           'Items',
        'avg_session_len': 'Avg. session len',
    })
)
display(
    disp_stats.style
    .format({'Interactions': '{:,.0f}', 'Sessions': '{:,.0f}', 'Items': '{:,.0f}', 'Avg. session len': '{:.1f}'})
    .set_caption('Table B1 — Processed dataset statistics')

    .hide(axis='index')
)

Dataset,Interactions,Sessions,Items,Avg. session len
Beauty,"33,488","4,243","3,625",7.9
Foursquare,"145,238","25,369","30,588",5.7
KuaiRec,"312,040","58,210","11,880",5.4
ML-1M,"575,281","37,420","3,706",15.4


In [4]:
# ── Table B3 — Full seen-target ranking results ───────────────────────────────
# Columns: one column per model × metric.  Rows: dataset × metric-shorthand.
METRICS_KEEP = ['hit@10', 'hit@20', 'ndcg@10', 'ndcg@20', 'mrr@10', 'mrr@20']
MODEL_ORDER  = ['RouteRec', 'SASRec', 'BERT4Rec', 'GRU4Rec', 'BPR']

sub = results[
    (results['split'] == 'test') &
    (results['metric'].isin(METRICS_KEEP))
]
pivot = (
    sub
    .pivot_table(index=['dataset_label', 'metric'], columns='model', values='value', aggfunc='mean')
    .reindex(columns=[m for m in MODEL_ORDER if m in sub['model'].unique()])
)

def highlight_best(row):
    best = row.max()
    return ['font-weight: bold' if v == best else '' for v in row]

display(
    pivot.style
    .apply(highlight_best, axis=1)
    .format('{:.4f}', na_rep='—')
    .set_caption('Table B3 — Full seen-target ranking results (test split, all reported cutoffs)')
)
annotate_demo(plt.gcf(), results)


<Figure size 832x624 with 0 Axes>